# AI Physics Tutor — Colab Benchmark Runner

MSc Thesis: **AI Assistant — A Multimodal Conversational Agent with Realistic Avatar for Physics Education**

Author: Kopalidis Thomas — MSc AI & DL, University of West Attica

---

## What this notebook does

1. **Setup** — Mount Google Drive, clone repo, install dependencies, download Ollama binary
2. **Prepare models** — Pull Ollama LLMs (gemma3:1b, qwen2.5:3b/7b/14b, llama3.1:8b)
3. **Build KB index** — Run `ingest_kb.py` and `build_index.py` (or restore from Drive)
4. **Smoke test** — Verify STT, RAG, LLM, TTS, Wav2Lip work end-to-end on this GPU
5. **Benchmark** (coming later) — Run the full experimental grid

## Prerequisites

- **Runtime**: GPU (T4 is enough for gemma/qwen 3b/7b; A100 needed for qwen 14b + Wav2Lip batching)
- **Account**: Google account with Drive access
- **Repo**: https://github.com/thomaskopalidis/thesis_ai_and_dl (public)

> **To change GPU type**: Runtime → Change runtime type → Hardware accelerator

## Cell 1 — Verify runtime and GPU

In [ ]:
import subprocess, sys, platform
print(f'Python: {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print('')
print('=== GPU ===')
try:
    out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True)
    print(out.strip())
except Exception as e:
    print(f'nvidia-smi failed: {e}  (are you on a GPU runtime?)')
print('')
print('=== Disk ===')
print(subprocess.check_output(['df', '-h', '/content'], text=True).strip())

## Cell 2 — Mount Google Drive

Drive gives us persistent storage so Ollama model downloads and benchmark results survive runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
THESIS_ROOT = '/content/drive/MyDrive/thesis_ai_and_dl'
os.makedirs(f'{THESIS_ROOT}/ollama_models', exist_ok=True)
os.makedirs(f'{THESIS_ROOT}/chroma_db', exist_ok=True)
os.makedirs(f'{THESIS_ROOT}/results', exist_ok=True)
os.makedirs(f'{THESIS_ROOT}/logs', exist_ok=True)
print(f'Thesis Drive folder: {THESIS_ROOT}')
!ls -la {THESIS_ROOT}

## Cell 3 — Clone the project repository

Clones into `/content/thesis_ai_and_dl` (fresh each runtime). Keep **big artifacts in Drive**, keep **code in GitHub**.

In [ ]:
import os, shutil
PROJECT_DIR = '/content/thesis_ai_and_dl'
REPO = 'https://github.com/thomaskopalidis/thesis_ai_and_dl.git'

if os.path.exists(PROJECT_DIR):
    print(f'Removing existing {PROJECT_DIR}')
    shutil.rmtree(PROJECT_DIR)

!git clone --depth 1 {REPO} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git log --oneline -5

## Cell 4 — Install Python dependencies

We pin versions known to work together (see docs/LAB_NOTEBOOK.md for the saga).
Colab usually has torch+cuda pre-installed; we skip reinstalling torch to avoid breaking the CUDA stack.

In [ ]:
# Core app deps (fast install, mostly already present on Colab)
!pip install -q \
    fastapi==0.115.0 'uvicorn[standard]==0.32.0' python-multipart==0.0.12 \
    pydantic==2.9.2 python-dotenv==1.0.1 \
    faster-whisper==1.0.3 \
    chromadb==0.5.15 sentence-transformers==3.2.1 \
    edge-tts==7.2.8 \
    requests==2.32.3 'numpy<2' \
    pymupdf==1.24.10 \
    soxr einops 'scikit-image' 'librosa==0.10.1' soundfile

# pkg_resources (older setuptools API needed by Wav2Lip's librosa import chain)
!pip install -q 'setuptools<70'

print('\n=== Verifying critical imports ===')
import importlib
for mod in ['torch', 'numpy', 'cv2', 'librosa', 'sentence_transformers',
            'chromadb', 'fastapi', 'faster_whisper', 'edge_tts',
            'soundfile', 'scipy', 'pkg_resources', 'einops']:
    try:
        m = importlib.import_module(mod)
        v = getattr(m, '__version__', '?')
        print(f'OK   {mod:28s} {v}')
    except Exception as e:
        print(f'FAIL {mod}: {type(e).__name__}: {str(e)[:80]}')

## Cell 5 — Install Ollama binary

Ollama is downloaded as a tgz, extracted into `/content/ollama_bin`. The server will be launched in Cell 7. Models live under `$THESIS_ROOT/ollama_models` so they survive runtime resets.

In [ ]:
import os, subprocess
OLLAMA_BIN_DIR = '/content/ollama_bin'
OLLAMA_URL = 'https://github.com/ollama/ollama/releases/download/v0.6.5/ollama-linux-amd64.tgz'

if not os.path.isfile(f'{OLLAMA_BIN_DIR}/bin/ollama'):
    !mkdir -p {OLLAMA_BIN_DIR}
    !wget -q -O /tmp/ollama.tgz {OLLAMA_URL}
    !tar -xzf /tmp/ollama.tgz -C {OLLAMA_BIN_DIR}
    !rm /tmp/ollama.tgz

os.environ['PATH'] = f'{OLLAMA_BIN_DIR}/bin:' + os.environ['PATH']
os.environ['OLLAMA_MODELS'] = f'{THESIS_ROOT}/ollama_models'
!ollama --version

## Cell 6 — Install ffmpeg (needed by STT and Wav2Lip output encoding)

In [ ]:
!apt-get -qq update && apt-get install -y -qq ffmpeg libgl1 libglib2.0-0 >/dev/null
!ffmpeg -version | head -1

## Cell 7 — Start Ollama server in background

Launches `ollama serve` and waits for the API to respond. We do NOT block the notebook — the server keeps running in the background.

In [ ]:
import subprocess, time, requests

# Kill any previous ollama server (idempotent)
subprocess.run(['pkill', '-f', 'ollama serve'], check=False)
time.sleep(1)

log_path = f'{THESIS_ROOT}/logs/ollama_serve.log'
env = os.environ.copy()
env['OLLAMA_MODELS'] = f'{THESIS_ROOT}/ollama_models'

with open(log_path, 'ab') as f:
    subprocess.Popen(['ollama', 'serve'], stdout=f, stderr=subprocess.STDOUT, env=env)

# Wait up to 30s for server to accept requests
for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=1)
        if r.ok:
            print(f'Ollama ready after {i+1}s')
            print('Available models:', [m['name'] for m in r.json().get('models', [])])
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print('[FAIL] Ollama did not start — check', log_path)

## Cell 8 — Pull the LLM models we will benchmark

This downloads ~22 GB total **once** (into Drive). Subsequent runtimes reuse the Drive-mounted models.

If Drive is slow, consider pulling into `/content` (local SSD) and copying only final results back to Drive.

In [ ]:
MODELS_TO_PULL = [
    'gemma3:1b',                       # 815 MB  — smoke test
    'qwen2.5:3b',                      # 1.9 GB
    'qwen2.5:7b',                      # 4.7 GB
    'llama3.1:8b',                     # 4.9 GB
    'qwen2.5:14b-instruct-q4_K_M',     # 9.0 GB — only if GPU has >= 20 GB VRAM
]

for m in MODELS_TO_PULL:
    print(f'\n=== Pulling {m} ===')
    !ollama pull {m} 2>&1 | tail -3

print('\n=== Final list ===')
!ollama list

## Cell 9 — Download OpenStax PDFs and build the KB index

The PDFs are not in git (too large). We download them once to Drive, run ingestion to produce `chunks.jsonl`, then build the ChromaDB index. All outputs live in Drive — next runtime restart will just reuse them.

In [ ]:
import os, urllib.request

KB_DIR = f'{PROJECT_DIR}/Knowledge_base'
SRC_DIR = f'{KB_DIR}/sources'
DRIVE_PDFS = f'{THESIS_ROOT}/openstax_pdfs'
os.makedirs(DRIVE_PDFS, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)

OPENSTAX = {
    'openstax_vol1.pdf': 'https://assets.openstax.org/oscms-prodcms/media/documents/UniversityPhysicsVolume1-WEB_Zwi59QO.pdf',
    'openstax_vol2.pdf': 'https://assets.openstax.org/oscms-prodcms/media/documents/UniversityPhysicsVolume2-WEB_oqeqFz9.pdf',
    'openstax_vol3.pdf': 'https://assets.openstax.org/oscms-prodcms/media/documents/UniversityPhysicsVolume3-WEB_8rXeDQZ.pdf',
}

for fname, url in OPENSTAX.items():
    drive_path = f'{DRIVE_PDFS}/{fname}'
    project_path = f'{SRC_DIR}/{fname}'
    if not os.path.isfile(drive_path):
        print(f'Downloading {fname}...')
        urllib.request.urlretrieve(url, drive_path)
    # Symlink from project dir into Drive copy — no duplication
    if not os.path.islink(project_path) and not os.path.isfile(project_path):
        os.symlink(drive_path, project_path)
    print(f'OK  {fname} ({os.path.getsize(drive_path)/1e6:.1f} MB)')

In [ ]:
# If a previously built chunks.jsonl exists in Drive, reuse it; otherwise re-ingest.
import os
DRIVE_CHUNKS = f'{THESIS_ROOT}/chunks.jsonl'
PROJECT_CHUNKS = f'{PROJECT_DIR}/Knowledge_base/processed/chunks.jsonl'
os.makedirs(f'{PROJECT_DIR}/Knowledge_base/processed', exist_ok=True)

if os.path.isfile(DRIVE_CHUNKS):
    !cp {DRIVE_CHUNKS} {PROJECT_CHUNKS}
    print(f'Reused chunks.jsonl from Drive ({os.path.getsize(PROJECT_CHUNKS)/1e6:.1f} MB)')
else:
    !cd {PROJECT_DIR} && python evaluation/ingest_kb.py --chunk-size 220 --overlap 32 --preview 2
    !cp {PROJECT_CHUNKS} {DRIVE_CHUNKS}
    print('Fresh ingestion → saved to Drive for reuse')

In [ ]:
# Build (or reuse) ChromaDB index. Store the DB in Drive so it persists.
import os
DRIVE_CHROMA = f'{THESIS_ROOT}/chroma_db'
PROJECT_CHROMA = f'{PROJECT_DIR}/chroma_db'

# Symlink project's chroma_db -> Drive's chroma_db so build_index writes to Drive directly
if os.path.exists(PROJECT_CHROMA) and not os.path.islink(PROJECT_CHROMA):
    import shutil; shutil.rmtree(PROJECT_CHROMA)
if not os.path.islink(PROJECT_CHROMA):
    os.symlink(DRIVE_CHROMA, PROJECT_CHROMA)

# Only build if the collection does not exist yet
import glob
if not glob.glob(f'{DRIVE_CHROMA}/*.sqlite3'):
    !cd {PROJECT_DIR} && python evaluation/build_index.py
else:
    print('ChromaDB index already present in Drive — skipping rebuild')
    !ls -la {DRIVE_CHROMA}

## Cell 10 — Smoke test the RAG

One retrieval query to confirm the index works and metadata is clean.

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)
from core.rag import retrieve_with_metadata

r = retrieve_with_metadata("What is Newton's second law?", top_k=3)
for i, (cid, meta, dist, doc) in enumerate(zip(r['ids'], r['metadatas'], r['distances'], r['documents']), 1):
    print(f'[{i}] dist={dist:.3f}  {meta.get("chapter")}  /  {meta.get("section")}')
    print(f'    {doc[:140]}...')
    print()

## Cell 11 — Smoke test the LLM

One question through gemma3:1b (fastest) to confirm Ollama + LLM wrapper work on GPU.

In [ ]:
import time, os
os.environ['OLLAMA_MODEL'] = 'gemma3:1b'

# Re-import config so it picks up the new env var
import importlib
import config
importlib.reload(config)
from core.llm import stream_llm_response

query = "What is Newton's second law?"
t0 = time.time()
parts = []
for token in stream_llm_response(query, model='gemma3:1b'):
    parts.append(token)
    if len(parts) == 1:
        print(f'[TIMING] first token: {time.time()-t0:.2f}s')
answer = ''.join(parts)
print(f'[TIMING] total: {time.time()-t0:.2f}s')
print(f'\nQ: {query}\nA: {answer}')

## Cell 12 — Smoke test TTS

Synthesise one short sentence to confirm edge-tts works in the sandbox.

In [ ]:
import asyncio, time
from core.tts import text_to_speech_stream

async def _test():
    t0 = time.time()
    nbytes = 0
    async for chunk in text_to_speech_stream('Hello from Google Colab.'):
        nbytes += len(chunk)
    print(f'[TIMING] TTS: {time.time()-t0:.2f}s  ({nbytes} bytes of mp3 audio)')

await _test()

## Cell 13 — Status summary

Everything above should have run without errors. If so, we are ready for the benchmark script (next step).

In [ ]:
import subprocess
print('=== Status ===')
print(f'Project:   {PROJECT_DIR}')
print(f'Drive:     {THESIS_ROOT}')
print(f'Ollama:    {OLLAMA_BIN_DIR}/bin/ollama')
print('')
print('=== Loaded models (VRAM) ===')
!ollama ps
print('')
print('=== GPU utilisation ===')
!nvidia-smi --query-gpu=name,utilization.gpu,memory.used,memory.total --format=csv,noheader